In [1]:
# Standard Libraries
import os
import sys
from collections import Counter

# Data manipulation and numerical libraries
import numpy as np
import pandas as pd

# Visualization library
import matplotlib.pyplot as plt

# Machine Learning utilities from scikit-learn
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix, f1_score,
    precision_recall_fscore_support, roc_curve, auc, make_scorer,
    precision_score, recall_score, roc_auc_score
)
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, label_binarize

# Bayesian Optimization library
from bayes_opt import BayesianOptimization

# Custom utilities for data processing and transformations
sys.path.append(os.path.dirname(os.getcwd()))  # Adjust path to include the top-level directory
from topicminer.utils.email_text_processing import read_json_to_dataframe
from topicminer.utils.statistical_transforms import train_test_split_utility, upsample_classes, encode_labels
from topicminer.config.config import PROCESSED_TEXT_COL, CATEGORIES_COL, TEST_SIZE

# Configuration for suppressing warnings
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='sklearn')

class AutoLabelEncoder:
    def __init__(self):
        self.encoder = LabelEncoder()
        self.is_encoded = False

    def fit_transform(self, y):
        """Automatically transforms non-numeric labels to numeric."""
        if not self.is_numeric(y):
            y = self.encoder.fit_transform(y)
            self.is_encoded = True
        return y

    def transform(self, y):
        """Transforms labels using the fitted encoder if they were encoded."""
        if self.is_encoded:
            return self.encoder.transform(y)
        return y

    def inverse_transform(self, y):
        """Converts numeric labels back to original if they were encoded."""
        if self.is_encoded:
            return self.encoder.inverse_transform(y)
        return y

    @staticmethod
    def is_numeric(y):
        """Check if the label array is numeric."""
        return issubclass(y.dtype.type, np.integer)

def rf_cv(params, data, targets, scoring='f1'):
    """
    Evaluate a RandomForest model with specific parameters based on a selectable scoring metric.

    Parameters:
        params: dict of parameters for RandomForest.
        data: feature data.
        targets: target data.
        scoring: str, default 'accuracy'
            The scoring metric to use. Options include 'accuracy', 'precision', 'recall', 'f1', 'roc_auc', and others.

    Returns:
        Mean of cross-validation scores.
    """
    # Define the estimator with parameters received
    estimator = RandomForestClassifier(
        n_estimators=int(params['n_estimators']),
        max_depth=int(params['max_depth']),
        min_samples_split=int(params['min_samples_split']),
        random_state=42
    )

    # Check if a custom scoring function needs to be used
    if scoring not in ['accuracy', 'roc_auc']:
        # Create a scorer from the make_scorer function
        if scoring == 'precision':
            scorer = make_scorer(precision_score, average='weighted')
        elif scoring == 'recall':
            scorer = make_scorer(recall_score, average='weighted')
        elif scoring == 'f1':
            scorer = make_scorer(f1_score, average='weighted')
        else:
            raise ValueError("Unsupported scoring method")
    else:
        scorer = scoring  # Use predefined scoring strings that sklearn recognizes

    # Return the mean of cross-validation scores
    return np.mean(cross_val_score(estimator, data, targets, scoring=scorer, cv=4))

def optimize_rf(data, targets,scoring='f1'):
    """
    Apply Bayesian Optimization to RandomForest parameters. This function aims to
    not only choose the best number of trees but also the optimal depth and minimum
    samples per split to balance between bias and variance effectively.

    Parameters:
        data: Training feature data.
        targets: Training target data.

    Returns:
        Dictionary with information about the best parameters and target score.
    """
    param_bounds = {
        'n_estimators': (10, 500),
        'max_depth': (3, 20),
        'min_samples_split': (2, 50)
    }

    optimizer = BayesianOptimization(
        f=lambda n_estimators, max_depth, min_samples_split: rf_cv({
            'n_estimators': n_estimators,
            'max_depth': max_depth,
            'min_samples_split': min_samples_split
        }, data, targets, scoring=scoring),
        pbounds=param_bounds,
        random_state=1,
        verbose=2
    )

    optimizer.maximize(init_points=2, n_iter=10)

    return optimizer.max

def train_rf_classifier(X_train_bal, y_train_bal, n_estimators, max_depth, min_samples_split):
    """
    Trains a RandomForest classifier using provided parameters.
    
    Parameters:
        X_train_bal: Balanced training feature data.
        y_train_bal: Balanced training label data.
        n_estimators: Optimized number of trees in the forest.
        max_depth: Optimized maximum depth of the tree.
        min_samples_split: Optimized minimum number of samples required to split an internal node.
    """
    # Initialize the RandomForestClassifier with provided parameters
    rf_clf = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=4,  # Assuming this is constant; adjust if needed
        random_state=42
    )

    # Fit the model on the balanced training data
    rf_clf.fit(X_train_bal, y_train_bal)

    return rf_clf

def score_rf_classifier(X_train_bal, X_test, y_train_bal, y_test, rf_clf):
    """
    Scores the RandomForest classifier and prints detailed metrics including accuracy,
    precision, recall, F1 score, and confusion matrix for both training and testing datasets.
    """
    # Predict and evaluate on the training set
    y_train_pred = rf_clf.predict(X_train_bal)
    train_accuracy = accuracy_score(y_train_bal, y_train_pred)
    train_precision, train_recall, train_f1, _ = precision_recall_fscore_support(y_train_bal, y_train_pred, average='weighted')

    # Predict and evaluate on the test set
    y_test_pred = rf_clf.predict(X_test)
    test_accuracy = accuracy_score(y_test, y_test_pred)
    test_precision, test_recall, test_f1, _ = precision_recall_fscore_support(y_test, y_test_pred, average='weighted')

    print("==== Training Metrics ====")
    print(f"Accuracy: {train_accuracy:.4f}")
    print(f"Precision: {train_precision:.4f}")
    print(f"Recall: {train_recall:.4f}")
    print(f"F1-Score: {train_f1:.4f}")

    print("\n==== Testing Metrics ====")
    print(f"Accuracy: {test_accuracy:.4f}")
    print(f"Precision: {test_precision:.4f}")
    print(f"Recall: {test_recall:.4f}")
    print(f"F1-Score: {test_f1:.4f}")

X_train_bal, X_test, y_train_bal, y_test = train_test_split_utility(upsampling_method='adasyn', use_lda_features=True)
best_params = optimize_rf(X_train_bal, y_train_bal)  # Assume this returns a dictionary of best parameters
rf_clf = train_rf_classifier(
    X_train_bal, y_train_bal,
    n_estimators=int(best_params['params']['n_estimators']),
    max_depth=int(best_params['params']['max_depth']),
    min_samples_split=int(best_params['params']['min_samples_split'])
)
score_rf_classifier(X_train_bal, X_test, y_train_bal, y_test, rf_clf)

Shapes - X_train: (2120, 1008), X_test: (1045, 1008), y_train: (2120,), y_test: (1045,)
|   iter    |  target   | max_depth | min_sa... | n_esti... |
-------------------------------------------------------------
| 1         | 0.8234    | 10.09     | 36.58     | 10.06     |
| 2         | 0.8936    | 8.14      | 9.044     | 55.25     |
| 3         | 0.8731    | 7.937     | 9.378     | 53.99     |
| 4         | 0.8929    | 8.541     | 8.382     | 57.73     |
| 5         | 0.9189    | 11.37     | 10.94     | 56.86     |
| 6         | 0.9001    | 10.35     | 13.98     | 58.48     |
| 7         | 0.9352    | 14.95     | 11.35     | 57.67     |
| 8         | 0.9379    | 15.95     | 8.116     | 55.78     |
| 9         | 0.9471    | 18.16     | 11.79     | 53.9      |
| 10        | 0.9529    | 20.0      | 10.2      | 57.94     |
| 11        | 0.9552    | 20.0      | 5.349     | 61.25     |
| 12        | 0.9535    | 20.0      | 9.879     | 64.85     |
==== Training Metrics ====
Accuracy: 0.9684
